<a href="https://colab.research.google.com/github/mariaaapetrovskaya/compling_nlp_course/blob/main/04_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [2]:
%pip install -q trafilatura stanza sentence-transformers scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 37.9 MB/s eta 0:00:00


In [3]:
import re
import zipfile
from pathlib import PurePosixPath

import numpy as np
import pandas as pd
import stanza
import trafilatura
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

languages = ["ar", "es", "ja", "ko", "tr"]

ZIP_PATH = "wikipedia_articles.zip"
ARTICLES_PER_LANGUAGE = 10
N_CLUSTERS = 5

In [4]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    html_files = [
        name
        for name in archive.namelist()
        if name.lower().endswith(".html")
        and len(PurePosixPath(name).parts) == 3
        and PurePosixPath(name).parts[1] in languages
    ]

article_counts = {
    lang: sum(PurePosixPath(name).parts[1] == lang for name in html_files)
    for lang in languages
}
print("Кол-во статей по языкам:")
display(pd.DataFrame(
    [{"language": lang, "articles": article_counts[lang]} for lang in languages]
))

incorrect = {l: c for l, c in article_counts.items() if c != ARTICLES_PER_LANGUAGE}
if incorrect:
    raise ValueError(
        f" нужно 10 статей на каждый язык"
        f"Проверенные количества: {article_counts}."
    )
print("Проверка пройдена")

Кол-во статей по языкам:


,language,articles
0,ar,10
1,es,10
2,ja,10
3,ko,10
4,tr,10


Проверка пройдена


In [5]:
articles = []
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for file_name in sorted(html_files):
        lang = PurePosixPath(file_name).parts[1]
        html = archive.read(file_name).decode("utf-8", errors="replace")
        text = trafilatura.extract(html)
        if not text or not text.strip():
            print("Не удалось извлечь:", file_name)
            continue
        text = re.sub(r"\s+", " ", text).strip()
        articles.append({
            "language": lang,
            "article": PurePosixPath(file_name).name,
            "text": text,
        })

articles_df = pd.DataFrame(articles)
print("Количество извл статей:", len(articles_df))
display(articles_df.groupby("language").size().to_frame("articles"))
display(articles_df[["language", "article", "text"]].head())

Количество извл статей: 50


,articles
language,
ar,10
es,10
ja,10
ko,10
tr,10


,language,article,text
0,ar,الجيرماكرانوليد - ويكيبيديا.html,الجيرماكرانوليد المظهر الجيرماكرانوليدات هي مج...
1,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...,العلاقات البوتسوانية الجنوب إفريقية | العلاقات...
2,ar,المعركة (قرية) - ويكيبيديا.html,المعركة (قرية) المظهر | المعركة | | |---|---| ...
3,ar,سياسة الصومال - ويكيبيديا.html,سياسة الصومال في أعقاب الحرب الأهلية وخلال عام...
4,ar,كارول كوسيرا - ويكيبيديا.html,كارول كوسيرا | | | |---|---| | (بالسلوفاكية: K...


In [6]:
nlp_models = {}
for lang in languages:
    print("Загружаем модель для языка:", lang)
    stanza.download(lang, processors="tokenize", verbose=False)
    nlp_models[lang] = stanza.Pipeline(lang, processors="tokenize", verbose=False)

sentences_data = []
for _, row in articles_df.iterrows():
    lang = row["language"]
    doc = nlp_models[lang](row["text"])
    for sentence in doc.sentences:
        s = re.sub(r"\s+", " ", sentence.text).strip()
        if s:
            sentences_data.append({
                "sentence": s,
                "language": lang,
                "article": row["article"],
            })

df = pd.DataFrame(sentences_data)
print("Всего предл:", len(df))
display(df.groupby("language").size().to_frame("sentences"))
display(df.head(10))

df.to_csv("sentences_dataset.csv", index=False, encoding="utf-8-sig")

Загружаем модель для языка: ar


models/tokenize/padt_nocharlm.pt: reconstructing file:   0%|          |  0.00B /  635kB            

models/tokenize/padt_nocharlm.pt: downloading bytes:           |  0.00B            

models/mwt/padt.pt: reconstructing file:   0%|          |  0.00B / 1.11MB            

models/mwt/padt.pt: downloading bytes:           |  0.00B            

...

models/mwt/imst.pt: reconstructing file:   0%|          |  0.00B /  611kB            

models/mwt/imst.pt: downloading bytes:           |  0.00B            

Всего предл: 4609


,sentences
language,
ar,30
es,2500
ja,1318
ko,295
tr,466


,sentence,language,article
0,الجيرماكرانوليد المظهر الجيرماكرانوليدات هي مج...,ar,الجيرماكرانوليد - ويكيبيديا.html
1,[2] الخصائص الكيميائية: - يتكوّن الهيكل الأساس...,ar,الجيرماكرانوليد - ويكيبيديا.html
2,العلاقات البوتسوانية الجنوب إفريقية | العلاقات...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
3,[1] لمحة تاريخية [عدل | عدل المصدر] يعود تاريخ...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
4,[3] صرَّح الرئيس البوتسواني كوت ماسيره عام 198...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
5,»[4] تأطرت العلاقات بين البلدين وأخذت طابعاً ر...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
6,[7] اتفق رئيسا البلدين في شهر نوفمبر من عام 20...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
7,[9] - ووقَّع الطرفين في عام 2005 ستة اتفاقيات ...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
8,www.tax-news.com. مؤرشف من الأصل في 2009-09-26...,ar,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...
9,المعركة (قرية) المظهر | المعركة | | |---|---| ...,ar,المعركة (قرية) - ويكيبيديا.html


In [7]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings = model.encode(df["sentence"].tolist(), show_progress_bar=True)
print("Размер масс.эмбеддингов:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

Размер масс.эмбеддингов: (4609, 768)


In [8]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(embeddings)

print("Кол-во кластеров:", df["cluster"].nunique())
display(pd.crosstab(df["language"], df["cluster"], margins=True))

Кол-во кластеров: 5


cluster,0,1,2,3,4,All
language,,,,,,
ar,19,1,0,0,10,30
es,773,1234,0,485,8,2500
ja,76,1,1217,23,1,1318
ko,70,0,15,2,208,295
tr,163,5,0,1,297,466
All,1101,1241,1232,511,524,4609


возможно такой результат из-за того, что короткие статьи на корейском попались, выборка неравномерная

In [9]:
ari = adjusted_rand_score(df["language"], df["cluster"])
print(f"Adjusted Rand Index (ARI): {ari:.4f}")

Adjusted Rand Index (ARI): 0.4557


In [10]:
cluster_language_counts = pd.crosstab(df["cluster"], df["language"])
cluster_to_language = cluster_language_counts.idxmax(axis=1).to_dict()

df["cluster_majority_language"] = df["cluster"].map(cluster_to_language)

errors = df[df["language"] != df["cluster_majority_language"]].copy()
print("Колво предложений отличающихс от кластеров:", len(errors))

if len(errors) > 0:
    examples = errors[
        ["sentence", "language", "cluster", "cluster_majority_language", "article"]
    ].head(3)
    display(examples)
else:
    print("ошибочных предложений нет")

Колво предложений отличающихс от кластеров: 603


,sentence,language,cluster,cluster_majority_language,article
0,الجيرماكرانوليد المظهر الجيرماكرانوليدات هي مج...,ar,4,tr,الجيرماكرانوليد - ويكيبيديا.html
1,[2] الخصائص الكيميائية: - يتكوّن الهيكل الأساس...,ar,0,es,الجيرماكرانوليد - ويكيبيديا.html
2,العلاقات البوتسوانية الجنوب إفريقية | العلاقات...,ar,4,tr,العلاقات البوتسوانية الجنوب إفريقية - ويكيبيدي...


ARI = 0.456 японский отделяется почти идеально, испанский разбивается на несколько кластеров из-за доминирования по размеру (скорее всего), а арабский почти полностью растворяется в других кластерах. Сильный дисбаланс классов, арабский как будто представлен всего 30 предложениями

## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga -
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче).
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги.
Обязательно покажите примеры, которые находятся по вашим запросам.


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [12]:
%pip install -q conllu spacy

In [13]:
import conllu
import spacy
from spacy.tokens import Doc
from spacy.matcher import DependencyMatcher
from collections import Counter
import pandas as pd

nlp_blank = spacy.blank("ru")

def conllu_to_doc(sent):
    words = [t for t in sent if isinstance(t["id"], int)]
    if not words:
        return None
    pos_map = {t["id"]: k for k, t in enumerate(words)}
    return Doc(
        nlp_blank.vocab,
        words=[t["form"] for t in words],
        spaces=[(t["misc"] or {}).get("SpaceAfter") != "No" for t in words],
        heads=[pos_map.get(t["head"], k) for k, t in enumerate(words)],
        deps=["ROOT" if t["head"] == 0 else t["deprel"] for t in words],
        pos=[t["upos"] for t in words],
        lemmas=[t["lemma"] for t in words],
        morphs=["|".join(f"{k}={v}" for k, v in (t["feats"] or {}).items()) for t in words],
    )

with open("ru_syntagrus-ud-train-b.conllu", encoding="utf-8") as f:
    docs = [d for d in map(conllu_to_doc, conllu.parse(f.read())) if d is not None]
print(len(docs))

24299


In [14]:
matcher = DependencyMatcher(nlp_blank.vocab)

matcher.add("AMOD", [[
    {"RIGHT_ID": "noun", "RIGHT_ATTRS": {"POS": "NOUN"}},
    {"LEFT_ID": "noun", "REL_OP": ">", "RIGHT_ID": "adj", "RIGHT_ATTRS": {"DEP": "amod"}},
]])

TARGETS = ["женщина", "мужчина", "писатель", "творец", "учительница"]
result = {t: Counter() for t in TARGETS}

for doc in docs:
    for _, (n_id, a_id) in matcher(doc):
        lemma = doc[n_id].lemma_.lower()
        if lemma in TARGETS:
            result[lemma][doc[a_id].lemma_.lower()] += 1

for t in TARGETS:
    c = result[t]
    total = sum(c.values())
    print(f"\n{t}  (всего определений: {total})")
    if total:
        for w, n in c.most_common(10):
            print(f"   {w:20s} {n}")
    else:
        print("   нет данных в train-b")


женщина  (всего определений: 44)
   красивый             6
   молодой              4
   беременный           4
   российский           2
   советский            2
   высокий              2
   кормить              1
   китайский            1
   танцевать            1
   работящий            1

мужчина  (всего определений: 7)
   обнаженный           1
   трудоспособный       1
   молодой              1
   здоровый             1
   неизвестный          1
   высокий              1
   светловолосый        1

писатель  (всего определений: 5)
   драматический        1
   будущий              1
   русский              1
   второстепенный       1
   знаменитый           1

творец  (всего определений: 0)
   нет данных в train-b

учительница  (всего определений: 1)
   симпатичный          1


In [16]:
for target in TARGETS:
    if not result[target]:
        continue
    top = [w for w, _ in result[target].most_common(3)]
    print(f"\n {target} (топ частые определения: {top})")
    shown = 0
    for doc in docs:
        for _, (n_id, a_id) in matcher(doc):
            n, a = doc[n_id], doc[a_id]
            if n.lemma_.lower() == target and a.lemma_.lower() in top:
                print(f"   {a.text} {n.text}:  {doc.text[:120]}")
                shown += 1
                if shown >= 2:
                    break
        if shown >= 2:
            break


 женщина (топ частые определения: ['красивый', 'молодой', 'беременный'])
   молодые женщины:  Наиболее подвержены риску молодые женщины: среди любительниц больших физических нагрузок, кому не было 30 лет, четверть 
   красивой женщиной:  Общаться с красивой женщиной - все равно что прыгать с парашютом. 

 мужчина (топ частые определения: ['обнаженный', 'трудоспособный', 'молодой'])
   обнаженный мужчина:  На нем изображен обнаженный мужчина с рогами, длинной бородой и привязанным сзади хвостом. 
   трудоспособные мужчины:  В ополчение вступали все трудоспособные мужчины и женщины в возрасте от 16 до 45 лет. 

 писатель (топ частые определения: ['драматический', 'будущий', 'русский'])
   драматического писателя:  Ольга Орлова: Не только "драматического писателя надо судить по законам, им самим над собою признанным", как считал Пушк
   будущих писателей:  С одной стороны, личная переписка может становиться фактом литературы (художественной или научной), если её ведут значит

 учительниц

2 устойчивые Adj + N коллокации (типа нескромный вопрос, сплошной ужас)

In [21]:
matcher = DependencyMatcher(nlp_blank.vocab)
from collections import Counter, defaultdict
matcher.add("ADJ_N", [[
    {"RIGHT_ID": "n", "RIGHT_ATTRS": {"POS": "NOUN"}},
    {"LEFT_ID": "n", "REL_OP": ">", "RIGHT_ID": "a",
     "RIGHT_ATTRS": {"POS": "ADJ"}},
]])

pairs = Counter()
by_noun = defaultdict(Counter)
by_adj = defaultdict(Counter)
contexts = defaultdict(list)

for doc in docs:
    for _, (n_id, a_id) in matcher(doc):
        n, a = doc[n_id], doc[a_id]
        al, nl = a.lemma_.lower(), n.lemma_.lower()
        if "VerbForm=Part" in a.morph:
            continue
        pairs[(al, nl)] += 1
        by_noun[nl][al] += 1
        by_adj[al][nl] += 1
        if len(contexts[(al, nl)]) < 3:
            contexts[(al, nl)].append(doc.text)

print("Всего пар (тип):", len(pairs))
print("Всего вхождений:", sum(pairs.values()))

Всего пар (тип): 23417
Всего вхождений: 34011


In [22]:
MIN_FREQ = 10
top = [(pair, c) for pair, c in pairs.most_common() if c >= MIN_FREQ]
print(f"Пар с частотой больше/равно {MIN_FREQ}: {len(top)}\n")
for (a, n), c in top[:30]:
    print(f"  {c:4d}   {a} {n}")

Пар с частотой больше/равно 10: 158

    57   прошлый год
    48   последний год
    47   2001 год
    44   2002 год
    43   xix век
    42   xx век
    39   последний время
    39   русский язык
    39   планерный спорт
    35   демонстрационный эффект
    33   уголовный дело
    31   железный дорога
    30   высший школа
    28   честный выборы
    28   данный случай
    27   верховный совет
    25   2000 год
    25   мировой война
    25   внутренний дело
    25   иностранный дело
    24   2009 год
    24   учебный заведение
    24   синхронный плавание
    22   долгий время
    22   сегодняшний день
    21   российский федерация
    21   высший образование
    20   народный депутат
    19   2007 год
    19   2006 год


In [23]:
for noun in ["вопрос", "ужас", "интерес", "стол", "время", "человек"]:
    if noun not in by_noun or by_noun[noun].total() < 20:
        continue
    print(f"\n{noun}  (всего вхождений: {by_noun[noun].total()})")
    for adj, c in by_noun[noun].most_common(10):
        print(f"   {c:4d}  {adj}")


вопрос  (всего вхождений: 97)
      8  данный
      4  спорный
      4  острый
      4  простой
      4  ключевой
      4  аграрный
      4  кадровый
      3  важный
      3  первый
      3  конкретный

интерес  (всего вхождений: 59)
      7  общечеловеческий
      5  особый
      3  общественный
      3  личный
      3  общий
      3  классовый
      2  собственный
      2  огромный
      2  коренной
      2  национальный

время  (всего вхождений: 241)
     39  последний
     22  долгий
     17  ближайший
     11  типичный
     10  советский
     10  свободный
      9  московский
      7  новый
      6  настоящий
      5  короткий

человек  (всего вхождений: 212)
     14  молодой
     11  советский
      7  нормальный
      6  хороший
      5  высокий
      5  живой
      5  здоровый
      4  умный
      4  больной
      4  конкретный


In [24]:
for (a, n), c in top[:3]:
    print(f"\n {a} {n}  (n={c})")
    for s in contexts[(a, n)]:
        print("  ", s[:130])


 прошлый год  (n=57)
   С 7 по 31 марта прошлого года в Сахаровском музее проходила художественная выставка "Запретное искусство-2006". 
   Официально около 20% школьников только по математике получили неудовлетворительную оценку на ЕГЭ в прошлом году. 
   По его словам, в начале прошлого года Плющенко еще был готов к диалогу. 

 последний год  (n=48)
   Хотя в последние годы демографическая проблема волновала ее, как замминистра здравоохранения, конечно, гораздо больше. 
   Неурожаи последних лет, капризы погоды, рост потребления продуктов в Индии и Китае - все это ведет к тому, что едоков становится б
   На протяжении последних 130 лет ученые неоднократно пытались прочитать надписи из долины Инда, но убедительной расшифровки никто н

 2001 год  (n=47)
   В 2001 году Усманов стал президентом и единственным спонсором Федерации фехтования России. 
   Когда Греция вступала в еврозону в 2001 году, она должна была выполнить условия по предельному размеру дефицита бюджета и размеру 
   В э

In [25]:
matcher = DependencyMatcher(nlp_blank.vocab)

matcher.add("ADJ_N", [[
    {"RIGHT_ID": "n", "RIGHT_ATTRS": {"POS": "NOUN"}},
    {"LEFT_ID": "n", "REL_OP": ">", "RIGHT_ID": "a",
     "RIGHT_ATTRS": {"POS": "ADJ"}},
]])

NOUN_STOP = {"год", "век", "раз", "день", "месяц", "число"}
ADJ_STOP = {"данный", "этот", "такой", "какой", "свой", "весь",
            "который", "наш", "ваш", "мой", "его", "её", "их"}

pairs = Counter()
by_noun = defaultdict(Counter)
contexts = defaultdict(list)

for doc in docs:
    for _, (n_id, a_id) in matcher(doc):
        n, a = doc[n_id], doc[a_id]
        al, nl = a.lemma_.lower(), n.lemma_.lower()

        if "VerbForm=Part" in a.morph:
            continue
        if nl in NOUN_STOP or al in ADJ_STOP:
            continue
        if any(ch.isdigit() for ch in a.text) or re.search(r"[IVX]+$", a.text):
            continue

        pairs[(al, nl)] += 1
        by_noun[nl][al] += 1
        if len(contexts[(al, nl)]) < 3:
            contexts[(al, nl)].append(doc.text)

MIN_FREQ = 10
top = [(p, c) for p, c in pairs.most_common() if c >= MIN_FREQ]
print(f"Пар с частотой ≥ {MIN_FREQ}: {len(top)}\n")
for (a, n), c in top[:30]:
    print(f"  {c:4d}   {a} {n}")

Пар с частотой ≥ 10: 126

    39   последний время
    39   русский язык
    39   планерный спорт
    35   демонстрационный эффект
    33   уголовный дело
    31   железный дорога
    30   высший школа
    28   честный выборы
    27   верховный совет
    25   мировой война
    25   внутренний дело
    25   иностранный дело
    24   учебный заведение
    24   синхронный плавание
    22   долгий время
    21   российский федерация
    21   высший образование
    20   народный депутат
    19   рабочий место
    19   детский дом
    19   большой количество
    19   второй половина
    18   золотой медаль
    18   местный житель
    18   правоохранительный орган
    18   морской вода
    17   ближайший время
    17   верховный суд
    17   культурный политика
    17   эстонский язык


In [26]:
for noun in ["вопрос", "интерес", "человек", "спорт", "дело", "эффект"]:
    if noun not in by_noun or by_noun[noun].total() < 10:
        continue
    total = by_noun[noun].total()
    print(f"\n{noun}  (всего: {total})")
    for adj, c in by_noun[noun].most_common(6):
        share = c / total
        print(f"   {c:3d}  ({share:.0%})  {adj}")


вопрос  (всего: 89)
     4  (4%)  спорный
     4  (4%)  острый
     4  (4%)  простой
     4  (4%)  ключевой
     4  (4%)  аграрный
     4  (4%)  кадровый

интерес  (всего: 59)
     7  (12%)  общечеловеческий
     5  (8%)  особый
     3  (5%)  общественный
     3  (5%)  личный
     3  (5%)  общий
     3  (5%)  классовый

человек  (всего: 210)
    14  (7%)  молодой
    11  (5%)  советский
     7  (3%)  нормальный
     6  (3%)  хороший
     5  (2%)  высокий
     5  (2%)  живой

спорт  (всего: 52)
    39  (75%)  планерный
     4  (8%)  российский
     3  (6%)  профессиональный
     1  (2%)  лыжный
     1  (2%)  олимпийский
     1  (2%)  авиационный

дело  (всего: 167)
    33  (20%)  уголовный
    25  (15%)  внутренний
    25  (15%)  иностранный
    12  (7%)  катынский
     5  (3%)  первый
     5  (3%)  международный

эффект  (всего: 68)
    35  (51%)  демонстрационный
     5  (7%)  побочный
     3  (4%)  экономический
     1  (1%)  особый
     1  (1%)  вредный
     1  (1%)  профилактическ

In [27]:
for (a, n), _ in top[:3]:
    print(f"{a} {n}")
    for s in contexts[(a, n)]:
        print("  ", s[:130])

последний время
   Если кому и доставалось от прессы в последнее время, так это главе Рособрнадзора Виктору Болотову. 
   Назначение Любови Глебовой еще раз подтвердило популярную в последнее время гипотезу активного прихода российских женщин во власть
   Продюсер Джаник Файзиев (фильмы "Турецкий гамбит" и "Адмиралъ") считает, что одна из главных проблем как раз в том, что "Черная мо
русский язык
   Во-первых, накануне единого госэкзамена по русскому языка задания по нему неожиданно выложил в интернете бесхитростный мальчик из 
   В 2008 году по русскому языку, литературе и математике сохраняется формула выставления отметок по принципу "плюс один балл". 
   Чтобы установить подобные закономерности, необязательно знать русский язык. 
планерный спорт
   Планерный спорт (планеризм) - вид активного отдыха и вид спорта, в котором пилоты летают без использования тяги на специализирован
   Во многих странах организуются региональные и национальные соревнования по планерному спорту, а также пр

Получается, в топе в основном терминологические сочетания типа планерный спорт, синхронное плавание, уголовное дело, прилагательное почти не варьируется.Это коллокации в строгом смысле

Вопрос и человек, тут прилагательные распределены равномерно, это свободные сочетания

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения?

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя.

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу?
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет?
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



После прочтения я подумала, что это покрытие более тысячи языков на самом деле не дает такой сильный "прогресс", как может показаться. Дело в источниках. Для многих низкоресурсных языков данных мало и они однородны (limited amount of data). В основном это Википедия, религиозные тексты или новости. Формально язык в датасете есть, но модель может не увидеть настоящую разговорную речь, тексты в соцсетях. Точно также модель не может увидеть и распознать ньюансы разных языков, которые были в этом домашнем задании. Если язык представлен одним регистром, это скорее присутствие, но не полное покрытие. Я бы не совмещала начало покрытия всех этих языков и реальное покрытие для некоторого числа языков. One pipeline to scale them all возможен как инфраструктура, не как единый набор критериев качества. Хороший многоязычный датасет должен быть скорее более контекстно чувствительным.
Мне кажется, стоит добавить, что в статье есть конкретная цифра, которая подтверждает эту мысль. Авторы пишут, что 70 процентов пар язык-письменность представляют собой больше половины документов из доменов, которые связаны с Библией или Википедией. Таким образом однородность данных еще и смещает оценку. Если тесты похожи на Википедию, модель может казаться лучше, чем она есть на живых текстах. Поэтому покрытие надо измерять и числом языков, и разнообразием самих источников внутри языка. Еще мне показалось интересным, что в статье сначала удаляются дубликаты, а потом часть повторяющихся данных возвращается через rehydration. То есть дубли это не всегда плохие данные. В низкоресурсных языках повторение может быть сигналом частотности или культурной каноничности. Но это уже расзмышления на другую тему. Главное, что масштаб не равен глубине, и хороший многоязычный датасет должен документировать именно полноту их представления (я уже не говорю о мультимодальном аспекте языков).